In [4]:
import pandas as pd
import plotly.express as px
import country_converter as coco

Reads the GDP tables from Wikipedia.

In [5]:
url = "https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(nominal)"

headers = {"User-Agent": "HW5-GDP-Analysis"}

Reads the table containing IMF data from Wikipedia and shows the first five rows.

In [6]:
gdp = pd.read_html(
    url,
    match="IMF",
    storage_options=headers
)[0]

gdp.head()

,Country/Territory,IMF (2026)[1],World Bank (2025)[6],United Nations (2024)[7]
0,World,128003776,118350166,111565144
1,United States,32642373,30769700,29298000
2,China[n 1],21546403,19498039,18743802
3,Germany,5471299,5050923,4659929
4,Japan,4341986,4435163,4026211


Keeps and renames the country and IMF GDP columns. It converts GDP values to numbers, removes missing values and the World total, and cleans the country names.

In [7]:
gdp = gdp[["Country/Territory", "IMF (2026)[1]"]].copy()

gdp = gdp.rename(columns={
    "Country/Territory": "country",
    "IMF (2026)[1]": "gdp"
})
gdp["gdp"] = pd.to_numeric(gdp["gdp"], errors="coerce")
gdp = gdp.dropna(subset=["gdp"])
gdp = gdp[gdp["country"] != "World"].copy()
gdp["country"] = gdp["country"].str.replace(
    r"\[.*?\]", "", regex=True
).str.strip()

gdp.head()

,country,gdp
1,United States,32642373.0
2,China,21546403.0
3,Germany,5471299.0
4,Japan,4341986.0
5,United Kingdom,4268446.0


Adds a region column based on each country's continent.

In [8]:
gdp["region"] = coco.convert(
    names=gdp["country"].tolist(),
    to="continent"
)

gdp.head()

,country,gdp,region
1,United States,32642373.0,America
2,China,21546403.0,Asia
3,Germany,5471299.0,Europe
4,Japan,4341986.0,Asia
5,United Kingdom,4268446.0,Europe


Creates an interactive bar plot with countries stacked within regions. Bar heights show IMF GDP values in million US dollars. The plot is saved as stacked_bar.html.

In [9]:
fig = px.bar(
    gdp,
    x="region",
    y="gdp",
    color="country",
    barmode="stack",
    title="GDP by Country within Regions",
    labels={
        "region": "Region",
        "gdp": "GDP (million US$)",
        "country": "Country"
    }
)

fig.write_html("stacked_bar.html")

Question 2

In [ ]:
import numpy as np
import plotly.graph_objects as go

Reads the brain region hierarchy and renames the columns.

In [13]:
url = "https://raw.githubusercontent.com/bcaffo/MRIcloudT1volumetrics/master/inst/extdata/multilevel_lookup_table.txt"

multilevel_lookup = pd.read_csv(url, sep = "\t").drop(['Level5'], axis = 1)
multilevel_lookup = multilevel_lookup.rename(columns = {
    "modify"   : "roi",
    "modify.1" : "level4",
    "modify.2" : "level3",
    "modify.3" : "level2",
    "modify.4" : "level1"})
multilevel_lookup = multilevel_lookup[['roi', 'level4', 'level3', 'level2', 'level1']]
multilevel_lookup.head()

,roi,level4,level3,level2,level1
0,SFG_L,SFG_L,Frontal_L,CerebralCortex_L,Telencephalon_L
1,SFG_R,SFG_R,Frontal_R,CerebralCortex_R,Telencephalon_R
2,SFG_PFC_L,SFG_L,Frontal_L,CerebralCortex_L,Telencephalon_L
3,SFG_PFC_R,SFG_R,Frontal_R,CerebralCortex_R,Telencephalon_R
4,SFG_pole_L,SFG_L,Frontal_L,CerebralCortex_L,Telencephalon_L


Selects Type 1, level 5 data for subject 127, adds the brain region hierarchy, and calculates volume proportions.

In [ ]:
id = 127
subjectData = pd.read_csv("https://raw.githubusercontent.com/smart-stats/ds4bio_book/main/book/assetts/kirby21AllLevels.csv")
subjectData = subjectData.loc[(subjectData.type == 1) & (subjectData.level == 5) & (subjectData.id == id)]
subjectData = subjectData[['roi', 'volume']]

subjectData = pd.merge(subjectData, multilevel_lookup, on = "roi")
subjectData = subjectData.assign(icv = "ICV")
subjectData = subjectData.assign(comp = subjectData.volume / np.sum(subjectData.volume))
subjectData.head()

,roi,volume,level4,level3,level2,level1,icv,comp
0,SFG_L,12926,SFG_L,Frontal_L,CerebralCortex_L,Telencephalon_L,ICV,0.009350
1,SFG_R,10050,SFG_R,Frontal_R,CerebralCortex_R,Telencephalon_R,ICV,0.007270
2,SFG_PFC_L,12783,SFG_L,Frontal_L,CerebralCortex_L,Telencephalon_L,ICV,0.009247
3,SFG_PFC_R,11507,SFG_R,Frontal_R,CerebralCortex_R,Telencephalon_R,ICV,0.008324
4,SFG_pole_L,3078,SFG_L,Frontal_L,CerebralCortex_L,Telencephalon_L,ICV,0.002227


Calculates the total volume for each connection between brain region levels.

In [19]:
import plotly.graph_objects as go

levels = ["icv", "level1", "level2", "level3", "level4", "roi"]

links = []

for i in range(len(levels) - 1):
    link = subjectData.groupby(
        levels[:i + 2]
    )["volume"].sum().reset_index()

    link["source"] = link[levels[:i + 1]].apply(tuple, axis=1)
    link["target"] = link[levels[:i + 2]].apply(tuple, axis=1)

    links.append(link[["source", "target", "volume"]])

links = pd.concat(links, ignore_index=True)

Creates a Sankey diagram from ICV through all five brain region levels and saves it as sankey.html.

In [20]:
nodes = list(dict.fromkeys(
    links["source"].tolist() + links["target"].tolist()
))

numbers = {node: i for i, node in enumerate(nodes)}

fig = go.Figure(go.Sankey(
    node=dict(
        label=[node[-1] for node in nodes],
        pad=15,
        thickness=15
    ),
    link=dict(
        source=links["source"].map(numbers).tolist(),
        target=links["target"].map(numbers).tolist(),
        value=links["volume"].tolist()
    )
))

fig.update_layout(
    title="MRICloud Subject 127: Type 1",
    width=1400,
    height=1800
)

fig.write_html("sankey.html")

My live Sankey diagram:

https://jiayichu77977-afk.github.io/149.628.-Data-Science-in-Python-HW5/sankey.html